In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import fisher_exact
from statsmodels.stats.multitest import multipletests
import altair as alt
from matplotlib.colors import ListedColormap
import matplotlib.patches as mpatches

# LOAD META

In [2]:
meta=pd.read_excel("/Users/jacobg/Dropbox/shared_leif/Jacob/metadata/wes_meta_JG202504.xlsx")
meta_wes = meta.dropna(subset=['wes tumor pre', 'wes tumor sg', 'wes extra FFT'], how='all')
meta_paired = meta_wes.dropna(subset='wes normal').set_index('new name', drop=False)
bad_samples = ['BIDMC2_pre', 'BIDMC3_post', 'BIDMC4_pre', 'BIDMC4_post', 'BIDMC5_post', 'BIDMC6_pre', 
               'BIDMC6_post', 'BIDMC7_post', 'DFC7_pre_FFT', 'DFC9_pre', 'DFC12_post', 'DFC19_pre', 
               'DFC19_pre', 'DFC20_post', 'DFC23_post', 'MGH3_post', 'MGH5_post', 'MGNS4_pre', 
               'NWH2_pre_FFT', 'MGNS5_post', 'DFC1_pre', 'DFC19_post', 'MGNS4_post', 'BIDMC3_post', 
               'DFC1_post']
for c in ['wes tumor pre', 'wes tumor sg', 'wes extra FFT', 'wes normal']:
    meta_paired[c] = meta_paired[c].replace(bad_samples, np.nan)


tumor_meta = pd.DataFrame({'tumor': pd.concat([meta_paired['wes tumor pre'], meta_paired['wes tumor sg'], meta_paired['wes extra FFT']])}).dropna()
tumor_meta = tumor_meta.reset_index().rename(columns={'new name': 'patient'}).set_index('tumor', drop=False)
tumor_meta['patient'] = ''
tumor_meta['timing'] = ''
tumor_meta['sg_response'] = ''
tumor_meta['final_response'] = ''
tumor_meta['plotname'] = ''

for patient in meta_paired.index:
    for group in ['wes tumor pre', 'wes tumor sg', 'wes extra FFT']:
        tumor = meta_paired.at[patient, group]
        if pd.isna(tumor):
            continue
        tumor_meta.at[tumor,'patient'] = patient
        tumor_meta.at[tumor,'sg_response'] = meta_paired.at[patient,'post sg response']
        tumor_meta.at[tumor,'final_response'] = meta_paired.at[patient,'final response(surgery)']
        if group in ['wes tumor pre', 'wes extra FFT']:
            tumor_meta.at[tumor,'plotname'] = meta_paired.at[patient,'wes pre plotname']
        else:
            tumor_meta.at[tumor,'plotname'] = meta_paired.at[patient,'wes post plotname']
        
tumor_meta['final_response_binary'] = tumor_meta['final_response'].apply(lambda x: 'pCR' if x == 'pCR' else 'RD')
tumor_meta['timing'] = tumor_meta['tumor'].apply(lambda x: 'post' if x.endswith("post") else 'pre')
tumor_meta['tissue_type'] = tumor_meta['tumor'].apply(lambda x: 'FFT' if x.endswith("FFT") else 'frozen')
tumor_meta.sort_values(by='patient', axis=0, inplace=True)

display(tumor_meta)
print(tumor_meta.shape)


,patient,tumor,timing,sg_response,final_response,plotname,final_response_binary,tissue_type
tumor,,,,,,,,
BIDMC1_pre,BIDMC1,BIDMC1_pre,pre,RD,RCB-I,P34R,RD,frozen
BIDMC3_pre,BIDMC3,BIDMC3_pre,pre,pCR,pCR,P38R,pCR,frozen
BIDMC5_pre,BIDMC5,BIDMC5_pre,pre,RD,RCB-II,P42R,RD,frozen
BIDMC7_pre,BIDMC7,BIDMC7_pre,pre,RD,RCB-I,P47R,RD,frozen
BIDMC8_pre,BIDMC8,BIDMC8_pre,pre,pCR,pCR,P49R,pCR,frozen
DFC1_pre_FFT,DFC1,DFC1_pre_FFT,pre,RD,RCB-I,P02R,RD,FFT
DFC10_pre,DFC10,DFC10_pre,pre,RD,pCR,P20R,pCR,frozen
DFC11_pre,DFC11,DFC11_pre,pre,pCR,pCR,P22R,pCR,frozen
DFC13_post,DFC13,DFC13_post,post,RD,RCB-II,P26T,RD,frozen


(45, 8)


# CNV SIGNATURES

## LOAD CNV SIGNAURES

In [3]:
path = '/Users/jacobg/Dropbox/shared_leif/Jacob/WES_2024/signatures/cnv_results/Assignment_Solution/Activities/Assignment_Solution_Activities.txt'
cnv_sigs = pd.read_csv(path, sep="\t")
cnv_sigs = cnv_sigs[cnv_sigs['Samples'].isin(bad_samples) == False]
cnv_sigs['patient'] = cnv_sigs.apply(lambda row: tumor_meta.at[row['Samples'], 'patient'], axis=1)
cnv_sigs['response'] = cnv_sigs.apply(lambda row: tumor_meta.at[row['Samples'], 'sg_response'], axis=1)
cnv_sigs['timing'] = cnv_sigs.apply(lambda row: tumor_meta.at[row['Samples'], 'timing'], axis=1)
cnv_sigs['sample_group'] = cnv_sigs.apply(lambda row: row['timing'] + '-' + row['response'], axis=1)
cnv_sigs['plotname'] = cnv_sigs.apply(lambda row: tumor_meta.at[row['Samples'], 'plotname'], axis=1)
cnv_sigs

,Samples,CN1,CN2,CN3,CN4,CN5,CN6,CN7,CN8,CN9,...,CN21,CN22,CN23,CN24,CN25,patient,response,timing,sample_group,plotname
0,BIDMC1_pre,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,BIDMC1,RD,pre,pre-RD,P34R
3,BIDMC3_pre,0,0,0,0,0,0,0,50,0,...,0,0,0,0,0,BIDMC3,pCR,pre,pre-pCR,P38R
7,BIDMC5_pre,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,BIDMC5,RD,pre,pre-RD,P42R
11,BIDMC7_pre,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,BIDMC7,RD,pre,pre-RD,P47R
12,BIDMC8_pre,0,0,0,0,0,0,0,55,0,...,0,0,0,0,0,BIDMC8,pCR,pre,pre-pCR,P49R
13,DFC10_pre,0,0,0,0,0,0,0,40,54,...,0,0,0,0,0,DFC10,RD,pre,pre-RD,P20R
14,DFC11_pre,0,32,0,0,0,0,0,0,0,...,0,0,0,0,0,DFC11,pCR,pre,pre-pCR,P22R
16,DFC13_post,10,31,0,0,0,0,0,0,0,...,0,0,0,0,0,DFC13,RD,post,post-RD,P26T
17,DFC13_pre,0,28,0,0,0,0,0,0,0,...,0,0,0,0,0,DFC13,RD,pre,pre-RD,P26R
18,DFC14_pre,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,DFC14,pCR,pre,pre-pCR,P27R


## ANALYSIS

In [4]:
# Define dictionaries and signature order
signature_order = [f'CN{x}' for x in range(1,26)]

cn_descriptions = {
    "CN1": "CN1 (Uniform Diploid)",
    "CN2": "CN2 (Uniform Tetraploid)",
    "CN3": "CN3 (Uniform Octoploid)",
    "CN4": "CN4 (Chromothripsis)",
    "CN5": "CN5 (Chromothripsis)",
    "CN6": "CN6 (Chromothripsis before WGD)",
    "CN7": "CN7 (Chromothripsis amplification)",
    "CN8": "CN8 (Chromothripsis amplification)",
    "CN9": "CN9 (Diploid chromosomal instability)",
    "CN10": "CN10 (LOH, 1×WGD)",
    "CN11": "CN11 (LOH, 2×WGD)",
    "CN12": "CN12 (Chromosomal instability + 1×WGD)",
    "CN13": "CN13 (Chromosomal LOH, 0×WGD)",
    "CN14": "CN14 (Chromosomal LOH, 1×WGD)",
    "CN15": "CN15 (Chromosomal LOH, 2×WGD)",
    "CN16": "CN16 (Chromosomal LOH + Het, 2×WGD)",
    "CN17": "CN17 (Homologous recombination deficiency)",
    "CN18": "CN18 (Unknown)",
    "CN19": "CN19 (Unknown)",
    "CN20": "CN20 (Unknown)",
    "CN21": "CN21 (Unknown)",
    "CN22": "CN22 (Artifacts)",
    "CN23": "CN23 (Artifacts)",
    "CN24": "CN24 (Artifacts)",
    "CN25": "CN25 (Deficient mismatch repair)"
}

cn_to_group = {
    "CN1": "Uniform Ploidy",
    "CN2": "Uniform Ploidy",
    "CN3": "Uniform Ploidy",
    "CN4": "Chromothripsis",
    "CN5": "Chromothripsis",
    "CN6": "Chromothripsis",
    "CN7": "Chromothripsis",
    "CN8": "Chromothripsis",
    "CN9": "Focal LOH",
    "CN10": "Focal LOH",
    "CN11": "Focal LOH",
    "CN12": "Focal LOH",
    "CN13": "Clonal LOH",
    "CN14": "Clonal LOH",
    "CN15": "Clonal LOH",
    "CN16": "Clonal LOH",
    "CN17": "HRD",
    "CN18": "Unknown",
    "CN19": "Unknown",
    "CN20": "Unknown",
    "CN21": "Unknown",
    "CN22": "Artifacts",
    "CN23": "Artifacts",
    "CN24": "Artifacts",
    "CN25": "Deficient mismatch repair"
}

# Create a list of signature names in the correct order
signature_name_order = []
for sig in signature_order:
    if sig in cn_descriptions:
        signature_name_order.append(cn_descriptions[sig])
    else:
        signature_name_order.append(sig)  # Keep original name if no description

# normalize
cnv_sigs_norm = cnv_sigs.copy()
signature_sums = cnv_sigs[signature_order].sum(axis=1)
for col in signature_order:
    cnv_sigs_norm[col] = cnv_sigs[col] / signature_sums

# Melt the dataframe
cnv_sigs_long = cnv_sigs_norm.melt(
    id_vars=['patient','plotname', 'timing', 'response', 'sample_group'],
    value_vars=signature_order,
    var_name='Signature',
    value_name='Value'
)

# Map signatures to their descriptions and groups
cnv_sigs_long['Signature Name'] = cnv_sigs_long['Signature'].map(cn_descriptions)
cnv_sigs_long['Signature Name'] = cnv_sigs_long['Signature Name'].fillna(cnv_sigs_long['Signature'])
cnv_sigs_long['Signature Group'] = cnv_sigs_long['Signature'].map(cn_to_group)

y_axis = alt.Axis(labelLimit=0, labelOverlap=False)

# Create the chart
chart = alt.Chart(cnv_sigs_long).mark_circle().encode(
    x= alt.Y('plotname:N', title='Sample'),
    y=alt.Y('Signature Name:N', sort=signature_name_order, axis=y_axis),
    size=alt.Size('Value:Q', title='Signature Contribution'),
    color='Signature Group:N',
)

# Configure the chart
chart = chart.facet(
    column=alt.Column('sample_group:N', title='Sample Group', sort=['pre-pCR','pre-RD','post-RD'])
).resolve_scale(
    x='independent'
).properties(
    title='Normalized Signatures by Sample Group'
).configure_view(
    stroke=None
).configure_axis(
    grid=False
).configure_legend(
    orient='bottom',
    labelFontSize=12,
    titleFontSize=14
)

# Display the chart
display(chart)
#chart.save('/Users/jacobg/Dropbox/shared_leif/neoSTAR A1 manuscript prep/exome_Jacob/plots/CN signatures dotplot.pdf')


alt.FacetChart(...)

## S1E

In [5]:
# Filter for values > 0 and within pre sample groups
filtered_data = cnv_sigs_long[(cnv_sigs_long['Value'] > 0) & (cnv_sigs_long['sample_group'].isin(['pre-pCR', 'pre-RD']))]
sample_counts = cnv_sigs_long.drop_duplicates(['plotname', 'sample_group'])[['sample_group']].value_counts().reset_index()
sample_counts.columns = ['sample_group', 'total_samples']

# Merge with sample counts to calculate proportions
grouped_data = filtered_data.groupby(['sample_group', 'Signature Name']).nunique()['plotname'].reset_index(name='count')
grouped_data = grouped_data.merge(sample_counts, on='sample_group')
grouped_data['Value'] = grouped_data['count'] / grouped_data['total_samples']

# Calculate the sum of proportions BEFORE any transformation
# This gives us our sorting key
sort_values = grouped_data.groupby('Signature Name')['Value'].sum().reset_index()
sort_values.columns = ['Signature Name', 'TotalProportion']

# Make pre-pCR values negative for the diverging chart
transformed_data = grouped_data.copy()
transformed_data['Value'] = transformed_data.apply(lambda x: -x['Value'] if x['sample_group'] == 'pre-pCR' else x['Value'], axis=1)

# Add display value column (absolute value for text display)
transformed_data['DisplayValue'] = transformed_data['Value'].abs()
transformed_data['DisplayText'] = (transformed_data['DisplayValue'] * 100).round().astype(int).astype(str) + '%'

# Merge the sorting values
transformed_data = transformed_data.merge(sort_values, on='Signature Name')

# Create a string representation of the sort order
signature_order = sort_values.sort_values('TotalProportion', ascending=False)['Signature Name'].tolist()

# Create the bar chart using the explicit sort order
bar_chart = alt.Chart(transformed_data).mark_bar().encode(
    y=alt.Y('Signature Name:N', title='Signature Name',
            sort=signature_order,  # Explicitly specify the sort order
            axis=alt.Axis(labelLimit=0)),
    x=alt.X('Value:Q', title='Proportion',
            scale=alt.Scale(domain=[-1, 1]),
            axis=alt.Axis(values=[-1, -0.75, -0.5, -0.25, 0, 0.25, 0.5, 0.75, 1],
                         labelExpr="abs(datum.value) == 0 ? '0%' : abs(datum.value * 100) + '%'")),
    color=alt.Color('sample_group:N', scale=alt.Scale(domain=['pre-pCR', 'pre-RD'], range=["#CC0C00FF", "#5C88DAFF"]))
)

# Create text labels with the same sort order
text_chart = alt.Chart(transformed_data).mark_text(
    align=alt.expr('datum.Value < 0 ? "right" : "left"'),
    baseline='middle', 
    dx=alt.expr('datum.Value < 0 ? -5 : 5'), 
    fontSize=12,
    fontWeight='bold'
).encode(
    y=alt.Y('Signature Name:N', sort=signature_order, axis=alt.Axis(labelLimit=0)),
    x=alt.X('Value:Q'),
    text='DisplayText:N',
    color=alt.value('black')
)

rule = alt.Chart(pd.DataFrame({'x': [0]})).mark_rule(color='black').encode(x='x:Q')
chart = bar_chart + text_chart

# # For debugging: print the calculated sort order
# print("Sorting order (highest proportion first):")
# for name, value in sort_values.sort_values('TotalProportion', ascending=False).values:
#     print(f"{name}: {value:.2f}")

final_chart = (chart + rule).properties(
    title='Percentage of CN Signatures pre-pCR vs pre-RD',
    width=500,
    height=300
).configure_axis(
    grid=True,
    gridDash=[3, 3]
).configure_axisX(
    title='← pre-RD | pre-pCR →'
).configure_view(
    stroke='transparent'
).configure_legend(
    title=None,
    orient='bottom',
    labelFontSize=12
)

display(final_chart)
#final_chart.save("/Users/jacobg/Dropbox/shared_leif/neoSTAR A1 manuscript prep/exome_Jacob/plots/CN signatures pcr vs rd.pdf")

alt.LayerChart(...)

In [6]:
# Get the raw counts we calculated previously
raw_counts = grouped_data[['sample_group', 'Signature Name', 'count', 'total_samples']]

# Create a function to calculate Fisher's exact test p-value for each signature
def calculate_p_value(signature_name):
    sig_data = raw_counts[raw_counts['Signature Name'] == signature_name]
    contingency = np.zeros((2, 2), dtype=int)
    
    # If a group doesn't have this signature, we need to handle that case
    for group in ['pre-pCR', 'pre-RD']:
        group_data = sig_data[sig_data['sample_group'] == group]
        if len(group_data) == 0:
            if group == 'pre-pCR':
                contingency[0, 0] = 0  # count with signature
                contingency[0, 1] = raw_counts[raw_counts['sample_group'] == group]['total_samples'].iloc[0]  # count without signature
            else:
                contingency[1, 0] = 0  # count with signature
                contingency[1, 1] = raw_counts[raw_counts['sample_group'] == group]['total_samples'].iloc[0]  # count without signature
        else:
            if group == 'pre-pCR':
                contingency[0, 0] = group_data['count'].iloc[0]  # count with signature
                contingency[0, 1] = group_data['total_samples'].iloc[0] - group_data['count'].iloc[0]  # count without signature
            else:
                contingency[1, 0] = group_data['count'].iloc[0]  # count with signature
                contingency[1, 1] = group_data['total_samples'].iloc[0] - group_data['count'].iloc[0]  # count without signature
    
    # Perform Fisher's exact test
    odds_ratio, p_value = stats.fisher_exact(contingency)
    return signature_name, p_value, odds_ratio, contingency

# Calculate p-values for all signatures
results = []
for signature in signature_order:
    result = calculate_p_value(signature)
    results.append(result)

# Create a DataFrame with the results
results_df = pd.DataFrame(results, columns=['Signature Name', 'P-value', 'Odds Ratio', 'Contingency Table'])

# Apply multiple testing correction using Benjamini-Hochberg method
results_df['Adjusted P-value (BH)'] = multipletests(results_df['P-value'], method='fdr_bh')[1]

# Sort by original signature order
results_df['Sort Order'] = results_df['Signature Name'].map({name: i for i, name in enumerate(signature_order)})
results_df = results_df.sort_values('Sort Order').drop('Sort Order', axis=1)

# Display the results
display(results_df[['Signature Name', 'P-value', 'Adjusted P-value (BH)', 'Odds Ratio']])
significant_signatures = results_df[results_df['P-value'] < 0.05]['Signature Name'].tolist()
print(f"\nSignificant signatures (p < 0.05): {', '.join(significant_signatures)}")
significant_adjusted = results_df[results_df['Adjusted P-value (BH)'] < 0.05]['Signature Name'].tolist()
print(f"\nSignificant after Benjamini-Hochberg correction: {', '.join(significant_adjusted)}")

,Signature Name,P-value,Adjusted P-value (BH),Odds Ratio
0,CN17 (Homologous recombination deficiency),0.306470,1.000000,2.357143
1,CN2 (Uniform Tetraploid),1.000000,1.000000,0.750000
2,CN12 (Chromosomal instability + 1×WGD),0.366625,1.000000,2.444444
3,CN18 (Unknown),0.303505,1.000000,3.833333
4,CN20 (Unknown),0.303505,1.000000,3.833333
5,CN8 (Chromothripsis amplification),1.000000,1.000000,1.050000
6,CN9 (Diploid chromosomal instability),0.035720,0.607236,0.000000
7,"CN13 (Chromosomal LOH, 0×WGD)",0.640870,1.000000,0.363636
8,CN1 (Uniform Diploid),0.640870,1.000000,0.363636
9,"CN14 (Chromosomal LOH, 1×WGD)",1.000000,1.000000,0.666667



Significant signatures (p < 0.05): CN9 (Diploid chromosomal instability)

Significant after Benjamini-Hochberg correction: 


## PRE POST PAIRED

In [7]:
cnv_sigs_long_paired = cnv_sigs_long[cnv_sigs_long['patient'].isin(['DFC2', 'DFC4', 'DFC13', 'MGNS6', 'MGH11', 'MGH12', 'DFC16', 'NWH3'])]


# Filter for values > 0 and within pre sample groups
filtered_data = cnv_sigs_long_paired[(cnv_sigs_long_paired['Value'] > 0) & (cnv_sigs_long_paired['sample_group'].isin(['pre-RD', 'post-RD']))]
sample_counts = cnv_sigs_long_paired.drop_duplicates(['plotname', 'sample_group'])[['sample_group']].value_counts().reset_index()
sample_counts.columns = ['sample_group', 'total_samples']

# Merge with sample counts to calculate proportions
grouped_data = filtered_data.groupby(['sample_group', 'Signature Name']).nunique()['plotname'].reset_index(name='count')
grouped_data = grouped_data.merge(sample_counts, on='sample_group')
grouped_data['Value'] = grouped_data['count'] / grouped_data['total_samples']

# Calculate the sum of proportions before transformation
sort_values = grouped_data.groupby('Signature Name')['Value'].sum().reset_index()
sort_values.columns = ['Signature Name', 'TotalProportion']

# Make pre-RD values negative for the diverging chart
transformed_data = grouped_data.copy()
transformed_data['Value'] = transformed_data.apply(lambda x: -x['Value'] if x['sample_group'] == 'pre-RD' else x['Value'], axis=1)

# Add display value column
transformed_data['DisplayValue'] = transformed_data['Value'].abs()
transformed_data['DisplayText'] = (transformed_data['DisplayValue'] * 100).round().astype(int).astype(str) + '%'

# Merge the sorting values
transformed_data = transformed_data.merge(sort_values, on='Signature Name')

# Create a string representation of the sort order
signature_order = sort_values.sort_values('TotalProportion', ascending=False)['Signature Name'].tolist()

# Create the bar chart using the explicit sort order
bar_chart = alt.Chart(transformed_data).mark_bar().encode(
    y=alt.Y('Signature Name:N', title='Signature Name',
            sort=signature_order,  # Explicitly specify the sort order
            axis=alt.Axis(labelLimit=0)),
    x=alt.X('Value:Q', title='Proportion',
            scale=alt.Scale(domain=[-1, 1]),
            axis=alt.Axis(values=[-1, -0.75, -0.5, -0.25, 0, 0.25, 0.5, 0.75, 1],
                         labelExpr="abs(datum.value) == 0 ? '0%' : abs(datum.value * 100) + '%'")),
    color=alt.Color('sample_group:N', scale=alt.Scale(domain=['pre-RD', 'post-RD'], range=["#5C88DAFF", "#FFCD00FF"]))
)

# Create text labels with the same sort order
text_chart = alt.Chart(transformed_data).mark_text(
    align=alt.expr('datum.Value < 0 ? "right" : "left"'),
    baseline='middle', 
    dx=alt.expr('datum.Value < 0 ? -5 : 5'), 
    fontSize=12,
    fontWeight='bold'
).encode(
    y=alt.Y('Signature Name:N', sort=signature_order, axis=alt.Axis(labelLimit=0)),
    x=alt.X('Value:Q'),
    text='DisplayText:N',
    color=alt.value('black')
)

rule = alt.Chart(pd.DataFrame({'x': [0]})).mark_rule(color='black').encode(x='x:Q')
chart = bar_chart + text_chart


final_chart = (chart + rule).properties(
    title='Percentage of CN Signatures Paired RD Samples',
    width=500,
    height=300
).configure_axis(
    grid=True,
    gridDash=[3, 3]
).configure_axisX(
    title='← pre-RD | post-pCR →'
).configure_view(
    stroke='transparent'
).configure_legend(
    title=None,
    orient='bottom',
    labelFontSize=12
)

display(final_chart)
#final_chart.save("/Users/jacobg/Dropbox/shared_leif/neoSTAR A1 manuscript prep/exome_Jacob/plots/CN signatures pre post paired.pdf")

alt.LayerChart(...)